# Waypoint — End-to-End Demo

This notebook walks through the full Waypoint workflow using real market data fetched from **yfinance** and **FRED**.

Sections:
1. Fetch real asset data (yfinance + FRED)
2. Portfolio construction
3. Expected Return (HistoricalMean)
4. Risk (SampleCovariance)
5. Efficient Frontier (Optimizer)
6. Wealth Simulation (MonteCarlo, inflation-adjusted withdrawals)

## 1. Fetch Real Data

We fetch 10 years of daily returns (2015–2024) for three assets via **yfinance**, and CPI from **FRED** to estimate long-run inflation.

| Asset | Symbol | Vendor |
|---|---|---|
| US Large Cap Equities | ^SPX | yfinance |
| Intl Developed Equities | EFA | yfinance |
| US Aggregate Bonds | AGG | yfinance |
| CPI YoY | CPIAUCSL | FRED |

**Prerequisites:**
- yfinance: `uv sync --extra yfinance`
- FRED: `uv sync --extra fred` + set `FRED_API_KEY` env var (free key at fred.stlouisfed.org)

Data is cached locally in parquet after the first fetch — re-running this notebook is instant.

In [1]:
from waypoint.catalog import US_LARGE_CAP, INTL_DEVELOPED, US_AGG_BONDS, CPI_YOY
from waypoint.data import fetch
from waypoint.portfolio import Portfolio
from waypoint.analysis.expected_return import ExpectedReturn
from waypoint.analysis.risk import Risk
from waypoint.analysis.optimizer import Optimizer
from waypoint.analysis.simulation import WealthSimulation
from waypoint.analysis.methods.returns import HistoricalMean
from waypoint.analysis.methods.risk import SampleCovariance
from waypoint.analysis.methods.simulation import MonteCarlo
from waypoint.constraints import LongOnly, SumToOne
from waypoint.cashflows import PeriodicCashflow
import numpy as np

START = "2015-01-01"
END   = "2024-12-31"
PERIODS_PER_YEAR = 252  # daily

# --- yfinance ---
equities = fetch(US_LARGE_CAP,    start=START, end=END)
intl     = fetch(INTL_DEVELOPED,  start=START, end=END)
bonds    = fetch(US_AGG_BONDS,    start=START, end=END)

for asset in [equities, intl, bonds]:
    print(f"{asset.name:30s}  {len(asset.returns):>5} days  "
          f"{asset.returns['date'].min()} → {asset.returns['date'].max()}")

# --- FRED (requires FRED_API_KEY) ---
# Falls back to 3% if the key is not set.
try:
    cpi = fetch(CPI_YOY, start=START, end=END)
    monthly_cpi = cpi.returns["returns"].to_numpy()
    annual_inflation = float((1 + monthly_cpi).prod() ** (12 / len(monthly_cpi)) - 1)
    print(f"\n{cpi.name:30s}  {len(cpi.returns):>5} months  "
          f"{cpi.returns['date'].min()} → {cpi.returns['date'].max()}")
    print(f"Estimated long-run annual inflation: {annual_inflation:.2%}")
except OSError as e:
    annual_inflation = 0.03
    print(f"\nFRED unavailable ({e}) — using default inflation rate of {annual_inflation:.0%}")

$^SPX: possibly delisted; no price data found  (1d 2015-01-01 -> 2015-01-02)
$EFA: possibly delisted; no price data found  (1d 2015-01-01 -> 2015-01-02)
$AGG: possibly delisted; no price data found  (1d 2015-01-01 -> 2015-01-02)


US Large Cap Equities            2515 days  2015-01-05 → 2024-12-31
Intl Developed Equities          2515 days  2015-01-05 → 2024-12-31
US Aggregate Bonds               2515 days  2015-01-05 → 2024-12-31

CPI YoY                           119 months  2015-02-01 → 2024-12-01
Estimated long-run annual inflation: 3.10%


## 2. Create Portfolio — 60/20/20 Initial Weights

Construct a portfolio with 60% US equities, 20% international equities, 20% bonds.

In [2]:
portfolio = Portfolio(
    slots={
        "US Equities": equities,
        "Intl Equities": intl,
        "US Bonds": bonds,
    },
    weights={
        "US Equities": 0.60,
        "Intl Equities": 0.20,
        "US Bonds": 0.20,
    },
    name="60/20/20 Global",
)

print("Portfolio weights:")
for name, weight in portfolio.weights.items():
    print(f"  {name}: {weight:.1%}")

print()
wide = portfolio.get_returns()
print(f"Aligned returns: {len(wide)} days  "
      f"{wide['date'].min()} → {wide['date'].max()}")
print(wide.head())

Portfolio weights:
  US Equities: 60.0%
  Intl Equities: 20.0%
  US Bonds: 20.0%

Aligned returns: 2515 days  2015-01-05 → 2024-12-31
shape: (5, 4)
┌────────────┬─────────────┬───────────────┬───────────┐
│ date       ┆ US Equities ┆ Intl Equities ┆ US Bonds  │
│ ---        ┆ ---         ┆ ---           ┆ ---       │
│ date       ┆ f64         ┆ f64           ┆ f64       │
╞════════════╪═════════════╪═══════════════╪═══════════╡
│ 2015-01-05 ┆ -0.018278   ┆ -0.023605     ┆ 0.002173  │
│ 2015-01-06 ┆ -0.008893   ┆ -0.011327     ┆ 0.002529  │
│ 2015-01-07 ┆ 0.01163     ┆ 0.011115      ┆ -0.00018  │
│ 2015-01-08 ┆ 0.017888    ┆ 0.013529      ┆ -0.001532 │
│ 2015-01-09 ┆ -0.008404   ┆ -0.004839     ┆ 0.002438  │
└────────────┴─────────────┴───────────────┴───────────┘


## 3. Expected Return — HistoricalMean

Estimate annualised expected returns using the arithmetic historical mean.

In [3]:
er_model = ExpectedReturn(method=HistoricalMean())
er_result = er_model.compute(
    portfolio,
    start=None,
    end=None,
    periods_per_year=PERIODS_PER_YEAR,
)

print(f"Method: {er_result.method_name}")
print("Per-asset annualised expected returns:")
for name, ret in er_result.per_asset.items():
    print(f"  {name}: {ret:.2%}")
print(f"Portfolio expected return: {er_result.portfolio:.2%}")

Method: HistoricalMean
Per-asset annualised expected returns:
  US Equities: 12.12%
  Intl Equities: 6.63%
  US Bonds: 1.42%
Portfolio expected return: 8.88%


## 4. Risk — SampleCovariance

Estimate the annualised covariance matrix and per-asset volatilities.

In [4]:
risk_model = Risk(method=SampleCovariance())
risk_result = risk_model.compute(
    portfolio,
    start=None,
    end=None,
    periods_per_year=PERIODS_PER_YEAR,
)

print(f"Method: {risk_result.method_name}")
print()
print("Annualised covariance matrix:")
print(risk_result.covariance)
print()
print("Per-asset annualised volatility:")
for name, vol in risk_result.volatilities.items():
    print(f"  {name}: {vol:.2%}")
print(f"Portfolio volatility: {risk_result.portfolio_volatility:.2%}")

Method: SampleCovariance

Annualised covariance matrix:
shape: (3, 3)
┌─────────────┬───────────────┬──────────┐
│ US Equities ┆ Intl Equities ┆ US Bonds │
│ ---         ┆ ---           ┆ ---      │
│ f64         ┆ f64           ┆ f64      │
╞═════════════╪═══════════════╪══════════╡
│ 0.031783    ┆ 0.02629       ┆ 0.000852 │
│ 0.02629     ┆ 0.029863      ┆ 0.001221 │
│ 0.000852    ┆ 0.001221      ┆ 0.00282  │
└─────────────┴───────────────┴──────────┘

Per-asset annualised volatility:
  US Equities: 17.83%
  Intl Equities: 17.28%
  US Bonds: 5.31%
Portfolio volatility: 13.91%


## 5. Efficient Frontier

Build the mean-variance efficient frontier using the `Optimizer`.
Constraints: long-only, weights sum to 1.

In [5]:
optimizer = Optimizer(
    return_model=ExpectedReturn(method=HistoricalMean()),
    risk_model=Risk(method=SampleCovariance()),
    constraints=[LongOnly(), SumToOne()],
)

frontier = optimizer.efficient_frontier(
    portfolio,
    start=None,
    end=None,
    periods_per_year=PERIODS_PER_YEAR,
    n_points=30,
)

print(f"Frontier points computed: {len(frontier.weights)}")

import polars as pl
summary = pl.DataFrame({
    "risk": frontier.risks,
    "expected_return": frontier.expected_returns,
})
print("\nFirst 5 frontier points (risk ascending):")
print(summary.head())

print()
sharpe_weights = frontier.optimal_sharpe(risk_free_rate=0.04)
print("Max Sharpe weights (risk-free rate = 4%):")
for name, w in sharpe_weights.items():
    print(f"  {name}: {w:.1%}")

Frontier points computed: 30

First 5 frontier points (risk ascending):
shape: (5, 2)
┌──────────┬─────────────────┐
│ risk     ┆ expected_return │
│ ---      ┆ ---             │
│ f64      ┆ f64             │
╞══════════╪═════════════════╡
│ 0.051984 ┆ 0.020648        │
│ 0.051984 ┆ 0.020648        │
│ 0.05201  ┆ 0.021625        │
│ 0.052582 ┆ 0.025312        │
│ 0.053879 ┆ 0.029           │
└──────────┴─────────────────┘

Max Sharpe weights (risk-free rate = 4%):
  US Equities: 100.0%
  Intl Equities: 0.0%
  US Bonds: -0.0%


In [6]:
# Plot the efficient frontier
fig = frontier.plot()
fig.show()

## 6. Wealth Simulation — Monte Carlo

Simulate 30-year wealth paths starting from $1,000,000 with a monthly $3,000 inflation-adjusted contribution.
The inflation rate is derived directly from the FRED CPI data fetched above.

In [7]:
# Monthly $3,000 contribution, grown annually by CPI-derived inflation
monthly_contribution = PeriodicCashflow(
    amount=3_000.0,
    frequency="monthly",
    mode="dollar",
    inflation_rate=annual_inflation,  # from FRED CPI above
)

wealth_sim = WealthSimulation(
    method=MonteCarlo(seed=42),
    cashflows=[monthly_contribution],
    horizon_years=30,
    initial_wealth=1_000_000.0,
    n_simulations=1000,
)

sim_result = wealth_sim.compute(
    portfolio,
    start=None,
    end=None,
    periods_per_year=PERIODS_PER_YEAR,
)

print(f"Simulation paths shape: {sim_result.paths.shape}")
print(f"  ({wealth_sim.n_simulations} simulations × "
      f"{wealth_sim.horizon_years * PERIODS_PER_YEAR + 1} periods)")
print()
s = sim_result.summary()
print("Terminal wealth summary (after 30 years):")
print(f"  5th percentile:  ${s['p5_terminal']:>12,.0f}")
print(f"  Median:          ${s['median_terminal']:>12,.0f}")
print(f"  95th percentile: ${s['p95_terminal']:>12,.0f}")

Simulation paths shape: (1000, 7561)
  (1000 simulations × 7561 periods)

Terminal wealth summary (after 30 years):
  5th percentile:  $   6,284,727
  Median:          $  17,693,680
  95th percentile: $  51,824,935


In [8]:
# Fan chart of wealth percentile paths
fig = sim_result.plot()
fig.show()